# Exercise 4.1 — One workflow, two classifiers (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day4/day4-01-frameworks.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

In Exercise 3.1 you predicted a number, blood glucose. Now the outcome is a
**class**: will this person have a heart attack or a stroke in the next ten
years, `yes` or `no`? The steps are
exactly the same: **specify** a model, **fit** it on the training patients, and
**predict** for new ones. Two things are new:

- the specification says the model is for **classification**;
- the prediction can be a **probability** for each class, as well as the class
  itself.

|  | Exercise 3.1: glucose, a number | Exercise 4.1: an event, a class |
|---|---|---|
| specify (R) | `linear_reg()`, `decision_tree()` with `set_mode("regression")` | `logistic_reg()`, `decision_tree()` with `set_mode("classification")` |
| specify (Python) | `LinearRegression()`, `DecisionTreeRegressor()` | `LogisticRegression()`, `DecisionTreeClassifier()` |
| fit | `fit()` / `.fit()` | the same |
| predict | a number | a class, or a probability for each class |

- **Part 1.** Predict an event from age with a logistic regression. Get the
  probability and the class for one person, see the patient on the fitted
  curve, and compare the predictions with what was measured in the test
  patients.
- **Part 2.** Add blood pressure and predict for the same person again. See
  the probability as a map over age and blood pressure.
- **Part 3.** Keep the same steps, training patients and predictors, and fit a
  decision tree instead. Follow the person to a leaf, read the probability and
  the class there, and compare the tree's map with the logistic regression's.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree

`data/cvd_risk.csv` holds 500 adults aged 40 to 80 seen at a health check:
their age, their systolic blood pressure in mmHg, and `cvd`, whether they had a
cardiovascular event (a heart attack or a stroke) in the following ten years,
`yes` or `no`. If you do not have it yet, [**download `cvd_risk.csv`**](data/cvd_risk.csv)
into your `data` folder.

In [ ]:
cvd = pd.read_csv("data/cvd_risk.csv")

cvd.describe()

We hold back every fifth person as the test set: 400 training patients and 100
test patients. Splitting by a fixed rule rather than at random gives both
languages the same 100 people, so this tab prints the same numbers as the R
tab. Every Day 4 exercise uses this split.

In [ ]:
is_test = cvd.index % 5 == 0               # every fifth person is held back
train = cvd[~is_test]
test  = cvd[is_test]

train["cvd"].value_counts()

### Part 1 — Predict an event from age

The same three steps as in Exercise 3.1. The class name says that this model
is for classification: `LogisticRegression` here, where Day 3 had
`LinearRegression`.

In [ ]:
# 1. Specify: logistic regression
log_model = LogisticRegression(C=np.inf)

# 2. Fit on the training patients: predictors (a table) and outcome
log_model.fit(train[["age"]], train["cvd"])

# the classes, and the learned parameters
print("classes:", log_model.classes_)
print("intercept:", log_model.intercept_)
print("weight for age:", log_model.coef_)

`C=np.inf` switches off a penalty that scikit-learn adds to logistic regression
by default; without it the parameters would not match the ones in the R tab.

The two parameters are not on the probability scale but on the **log-odds**
scale. Exercise 4.2 shows how they turn into a probability. Here we let
scikit-learn do it.

#### Task 1a — A 70-year-old

`.predict()` gives the **class**. `.predict_proba()` gives a **probability** for
each class instead, one column per class in the order of `classes_`: `no`
first, then `yes`.

> **Stuck?**
>
> The method's name is `predict` followed by an underscore and a short word for
> probability.

> **Worked answer**

In [ ]:
person = pd.DataFrame({"age": [70]})

print(log_model.predict_proba(person))   # a probability for each class: no, yes
print(log_model.predict(person))  # the class

> The probability of `no` is 0.453 and of `yes` 0.547: they always add up to 1.
> The class is `yes`, because the probability of `yes` is above 0.5.

The class is the probability cut at 0.5: `yes` if $\hat{P}(\text{yes}) > 0.5$,
`no` otherwise. The red point is the 70-year-old, on the fitted curve:

In [ ]:
curve = pd.DataFrame({"age": np.arange(40, 80.5, 0.5)})
curve_prob = log_model.predict_proba(curve)[:, 1]
rng = np.random.default_rng(1)

fig, ax = plt.subplots()
ax.scatter(train["age"] + rng.uniform(-0.3, 0.3, len(train)),
           (train["cvd"] == "yes") + rng.uniform(-0.03, 0.03, len(train)),
           alpha=0.3, color="steelblue")
ax.plot(curve["age"], curve_prob, color="red", linewidth=2)
ax.axhline(0.5, linestyle="--", color="grey")
ax.scatter([70], log_model.predict_proba(person)[:, 1], color="red", s=80, zorder=3)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Probability of an event")
ax.set_title("Training patients (0 = no, 1 = yes), the fitted curve, and the 70-year-old")
plt.show()

> **Think about it**
>
> 1. The model says `yes` for this person. How sure is it?
> 2. At about what age does the predicted class change from `no` to `yes`?

> **Worked answer**
>
> 1. Hardly sure at all: 0.547 is only just above 0.5. The class `yes` on its own
>    hides that; the probability shows it.
> 2. Where the curve crosses the dashed line at 0.5: at about 68 years (67.7).
>    Below that the class is `no`, above it `yes`.

#### Task 1b — Predictions against what happened

The test patients were not used to fit the model. Predict both the probability
of `yes` and the class for each of them, and put them next to what happened.

In [ ]:
test_results = test.copy()
test_results["prob_yes"] = log_model.predict_proba(test[["age"]])[:, 1]
test_results["predicted"] = log_model.predict(test[["age"]])

test_results.head(10)

How often is the predicted class right? Compare it with a "model" that always
says `no`, the most common class.

> **Stuck?**
>
> Compare the predicted class with the column that holds what happened.

> **Worked answer**

In [ ]:
print((test_results["predicted"] == test_results["cvd"]).mean())   # accuracy of the model
print((test_results["cvd"] == "no").mean())                          # always saying no

> The model is right for 81% of the test patients. Always saying `no` is right
> for 66%, because 66 of the 100 had no event.

In [ ]:
fig, ax = plt.subplots()
ax.axhline(0.5, linestyle="--", color="grey")
for cls, colour in [("no", "steelblue"), ("yes", "darkorange")]:
    rows = test_results[test_results["cvd"] == cls]
    ax.scatter(rows["age"] + rng.uniform(-0.3, 0.3, len(rows)), rows["prob_yes"],
               color=colour, alpha=0.7, label=cls)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Predicted probability of an event")
ax.legend(title="What happened")
ax.set_title("Test patients: predicted probability, coloured by what happened")
plt.show()

> **Think about it**
>
> 1. Every point lies on the same curve. What has the model got wrong for the
>    orange points below the dashed line, and for the blue points above it?
> 2. Which is more useful to a clinician: the class, or the probability?

> **Worked answer**

In [ ]:
pd.crosstab(test_results["predicted"], test_results["cvd"],
            rownames=["predicted"], colnames=["happened"])

> 1. The orange points below the line had an event but were called `no`: 12
>    people. The blue points above it had none but were called `yes`: 7 people.
>    With age as the only predictor, two people of the same age get the same
>    probability, whatever else is true of them.
> 2. The probability. A person at 0.52 and a person at 0.72 are both `yes`, but
>    they are not equally worrying. The cut-off of 0.5 is a choice, not part of
>    the model; Exercise 4.2 is about choosing it.

### Part 2 — Add blood pressure

The same kind of model, with a second predictor:

In [ ]:
log_model_2 = LogisticRegression(C=np.inf)
log_model_2.fit(train[["age", "systolic_bp"]], train["cvd"])

print("intercept:", log_model_2.intercept_)
print("weights (age, systolic_bp):", log_model_2.coef_)

#### Task 2a — The same person, now with a blood pressure of 130

> **Stuck?**
>
> The person now needs a value for both predictors. Ask for the probabilities
> exactly as in Task 1a.

> **Worked answer**

In [ ]:
person = pd.DataFrame({"age": [70], "systolic_bp": [130]})

print(log_model_2.predict_proba(person))
print(log_model_2.predict(person))

> The probability of an event is now 0.487, and the class is `no`.

With two predictors, the probability is a **map** over age and blood pressure.
The black line is where the probability is exactly 0.5: above it the class is
`yes`, below it `no`. Circles are test patients without an event, triangles
test patients with one, and the red point is the 70-year-old.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# every combination of age and blood pressure on a fine grid
A, S = np.meshgrid(np.arange(40, 80.5, 0.5), np.arange(80, 201, 1))
grid = pd.DataFrame({"age": A.ravel(), "systolic_bp": S.ravel()})
blue_orange = LinearSegmentedColormap.from_list("bo", ["steelblue", "white", "darkorange"])

def prob_map(model, title):
    prob = model.predict_proba(grid)[:, 1].reshape(A.shape)
    fig, ax = plt.subplots()
    im = ax.pcolormesh(A, S, prob, cmap=blue_orange, vmin=0, vmax=1, shading="auto")
    ax.contour(A, S, prob, levels=[0.5], colors="black")
    no, yes = test[test["cvd"] == "no"], test[test["cvd"] == "yes"]
    ax.scatter(no["age"], no["systolic_bp"], marker="o", facecolors="none",
               edgecolors="black", s=20, label="test: no")
    ax.scatter(yes["age"], yes["systolic_bp"], marker="^", color="black", s=20,
               label="test: yes")
    ax.scatter([70], [130], color="red", s=80, zorder=3, label="70-year-old")
    fig.colorbar(im, ax=ax, label="Probability of an event")
    ax.set_xlabel("Age (years)")
    ax.set_ylabel("Systolic blood pressure (mmHg)")
    ax.set_title(title)
    ax.legend(loc="lower left", fontsize=8)
    plt.show()

prob_map(log_model_2, "Logistic regression: age and blood pressure")

> **Think about it**
>
> 1. The line where the class changes is straight but tilted. What does the tilt
>    tell you?
> 2. Adding blood pressure moved this person from 0.547 to 0.487, and the class
>    from `yes` to `no`. Did it change much about this person's prediction?

> **Worked answer**
>
> 1. The higher the blood pressure, the younger the age at which the model
>    switches to `yes`: at 140 mmHg the line is at about 65 years, at 160 mmHg at
>    about 55. Both predictors push the probability the same way, and neither
>    decides alone. Exercise 4.3 works this line out from the coefficients.
> 2. Very little: the probability moved by 0.06. The class flipped only because
>    the person sits right next to the line. Near 0.5 a small change in the
>    probability can change the class; the probability shows that, the class
>    does not. On the 100 test patients the model with both predictors is right
>    83% of the time, against 81% with age alone.

### Part 3 — The same steps, a decision tree

Only the specification changes. The training patients, the predictors and the
`.fit()`, `.predict()` and `.predict_proba()` calls stay the same.

In [ ]:
# 1. Specify: a classification tree with at most five leaves
tree_model = DecisionTreeClassifier(max_leaf_nodes=5, random_state=0)

# 2. Fit on the same training patients, with the same predictors
tree_model.fit(train[["age", "systolic_bp"]], train["cvd"])

fig, ax = plt.subplots(figsize=(11, 6))
plot_tree(tree_model, feature_names=["age", "systolic_bp"], class_names=["no", "yes"],
          filled=False, impurity=False, proportion=True, precision=2, ax=ax)
plt.show()

`max_leaf_nodes=5` allows at most five leaves, which gives the same tree as the
R tab. Without it the tree keeps splitting until every leaf holds people of one
class only: 149 leaves, which fit the training patients almost one by one and
are right for only 66% of the test patients, no better than always saying `no`.

Each box asks a yes/no question. **True goes left, False goes right.** In each
leaf at the bottom, `samples` is the percentage of all training patients who
ended up there, `value` gives the share of them in each class (`no`, `yes`),
and `class` is the predicted class. The share of `yes` is the leaf's
probability of an event.

#### Task 3a — Follow the person

Start at the top with the 70-year-old with a blood pressure of 130. At each
question, go left or right. Which leaf do you reach? What probability and what
class does it give?

> **Worked answer**
>
> `age <= 53.5`? False, go right. `systolic_bp <= 135.5`? True, go left.
> `age <= 71.5`? True, go left. The leaf holds 22.5% of the training patients
> (90 of 400); 21% of them had an event, so the probability is **0.21** and the
> class is `no`.

Check it with `.predict_proba()` and `.predict()`:

In [ ]:
print(tree_model.predict_proba(person))
print(tree_model.predict(person))

The same map as for the logistic regression:

In [ ]:
prob_map(tree_model, "Decision tree: age and blood pressure")

How often is each model right on the test patients?

In [ ]:
pd.DataFrame({
    "model": ["logistic, age", "logistic, age + BP", "tree, age + BP",
              "always says no"],
    "accuracy": [log_model.score(test[["age"]], test["cvd"]),
                 log_model_2.score(test[["age", "systolic_bp"]], test["cvd"]),
                 tree_model.score(test[["age", "systolic_bp"]], test["cvd"]),
                 (test["cvd"] == "no").mean()]})

`.score()` is the accuracy: the share of people whose predicted class is right.

> **Think about it**
>
> 1. The logistic regression gives every person their own probability. How many
>    different probabilities can the tree give, and why?
> 2. Compare the two maps. What shape is the region where each model says `yes`?
> 3. The three models gave the 70-year-old 0.55, 0.49 and 0.21. Their accuracies
>    on the test patients are close. What does that tell you?

> **Worked answer**
>
> 1. Five, one per leaf: 0.14, 0.21, 0.12, 0.70 and 0.67. Every person in the
>    same leaf gets the same probability, the share of training patients in that
>    leaf who had an event. Everyone up to 53.5 gets 0.14, whatever their blood
>    pressure.
> 2. The logistic regression changes smoothly, and its `yes` region is
>    everything above a straight line. The tree's map is made of flat rectangles:
>    it says `yes` for anyone over 53.5 with a blood pressure over 135.5, and for
>    anyone over 71.5 with a blood pressure between 114.5 and 135.5.
> 3. Models can agree on most classes and still give quite different
>    probabilities for a single person. Accuracy only counts classes; it says
>    nothing about how good the probabilities are. With 100 test patients, a difference of
>    two people (83% against 81%) is far too small to pick a winner.

> **Learning points**
>
> - The steps are the same as in Exercise 3.1: specify, fit on the training
>   patients, predict for new ones. For a class instead of a number, only the
>   estimator changes: `LogisticRegression` and `DecisionTreeClassifier`.
> - `.predict()` gives the class; `.predict_proba()` gives a probability for each
>   class. The class is the probability cut at 0.5.
> - The probability carries more information than the class: near 0.5 a small
>   change flips the class.
> - A logistic regression's probability changes smoothly and its classes are
>   split by a straight line; a tree gives one probability per leaf and splits
>   the classes into rectangles.